<a href="https://colab.research.google.com/github/Zattyla/K-RVC-Colab/blob/main/K-RVC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RVC Voice Converter (v2.0)
*State-of-the-art AI Voice Transformation*

### 🚀 Pre-requisites:
1.  **Audio from Colab A:** You must have a clean vocal file (wav) from the **UVR Elite Separator**.
2.  **GPU Check:** Go to `Runtime` -> `Change runtime type` and select **T4 GPU**.

### 🛠️ Execution Flow:
1.  **Run Cell 1:** Setup the Applio engine (Numpy 1.26 optimized).
2.  **Run Cell 2:** Download RMVPE and base pretrained models.
3.  **Run Cell 3:** Paste your RVC Model link (.zip) and name it.
4.  **Upload Vocal:** Upload your clean vocal to the `/inputs` folder on the left.
5.  **Run Cell 4:** Select your pitch (0 for same gender, +12 for Male to Female) and convert!

---

In [ ]:
# @title 1. RVC Engine Setup
import os

# Create workspace folders
os.makedirs("/content/inputs", exist_ok=True)
os.makedirs("/content/outputs", exist_ok=True)

print("⏳ Installing RVC Engine (Applio)...")
# Force compatible numpy version first
!pip install numpy==1.26.4

# Clone and install Applio
if not os.path.exists("/content/Applio"):
    !git clone https://github.com/IAHispano/Applio.git
    %cd /content/Applio
    !pip install -r requirements.txt
    !pip install -q faiss-cpu fairseq pyworld pydub

print("✅ RVC Engine is ready.")

⏳ Installing RVC Engine (Applio)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 41.7 MB/s eta 0:00:00


In [ ]:
# @title 2. Download RVC Essentials (RMVPE)
print("⏬ Downloading RMVPE Pitch Predictor...")

# RMVPE is the best method for pitch extraction
!aria2c --console-log-level=error -c -x 16 -s 16 "https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main/rmvpe.pt" -d "/content/Applio/rvc/models/predictors" -o rmvpe.pt

# Download Base Pretrained Models
!aria2c --console-log-level=error -c -x 16 -s 16 "https://huggingface.co/IAHispano/Applio/resolve/main/Resources/Pretraineds/f0G40k.pth" -d "/content/Applio/rvc/models/pretraineds" -o f0G40k.pth

print("✅ Essential models downloaded.")

In [ ]:
# @title 3. Download Voice Model
import zipfile

# @markdown ### 📂 Model Settings
# @markdown Paste the link to the RVC Model (.zip):
MODEL_URL = "" # @param {type:"string"}
# @markdown Name your model (No spaces):
MODEL_NAME = "My_AI_Voice" # @param {type:"string"}

if MODEL_URL:
    model_zip = f"/content/Applio/logs/{MODEL_NAME}.zip"
    model_dir = f"/content/Applio/logs/{MODEL_NAME}"
    os.makedirs(model_dir, exist_ok=True)

    print(f"⏳ Downloading {MODEL_NAME}...")
    !aria2c --console-log-level=error -c -x 16 -s 16 "{MODEL_URL}" -d "/content/Applio/logs" -o "{MODEL_NAME}.zip"

    if os.path.exists(model_zip):
        with zipfile.ZipFile(model_zip, 'r') as zip_ref:
            zip_ref.extractall(model_dir)
        print(f"✅ Model '{MODEL_NAME}' is ready!")
    else:
        print("❌ Download failed.")

In [ ]:
# @title 4. Run Voice Conversion
%cd /content/Applio

# @markdown ### 🎤 Input Settings
# @markdown Filename of the vocal uploaded to '/inputs':
INPUT_FILE = "vocal.wav" # @param {type:"string"}

# @markdown ### ⚙️ Conversion Settings
# @markdown Pitch: 0 (Original), 12 (Male to Female), -12 (Female to Male)
PITCH = 0 # @param {type:"slider", min:-24, max:24, step:1}
# @markdown Model name used in Step 3:
SELECTED_MODEL = "My_AI_Voice" # @param {type:"string"}

input_path = f"/content/inputs/{INPUT_FILE}"
output_path = f"/content/outputs/AI_{INPUT_FILE}"

if os.path.exists(input_path):
    print(f"🚀 Converting {INPUT_FILE} to {SELECTED_MODEL}...")

    !python infer.py \
        --input_path "{input_path}" \
        --output_path "{output_path}" \
        --model_name "{SELECTED_MODEL}" \
        --transpose {PITCH} \
        --f0_method "rmvpe" \
        --index_rate 0.7 \
        --volume_envelope 1

    print(f"✨ Conversion complete! File saved in /outputs/AI_{INPUT_FILE}")
else:
    print(f"❌ Error: {INPUT_FILE} not found in /inputs.")

In [ ]:
# @title 5. Cleanup & GPU Refresh
import torch, gc
torch.cuda.empty_cache()
gc.collect()
print("✨ GPU Memory cleared. Ready for the next conversion!")

# 🛠️ Troubleshooting Guide (RVC Edition)

### 1. ❌ "ModuleNotFoundError: No module named 'numpy'"
* **Solution:** Restart the session (`Runtime` -> `Restart Session`) and run **Cell 1** again. This happens if Colab tries to use the default numpy instead of the version we installed.

### 2. ❌ Voice sounds "robotic" or "distorted"
* **Solution:** * Ensure you used **Cell 4 (De-reverb)** in Colab A before bringing the file here.
    * Adjust the **Pitch** (Step 4). If a woman is singing a man's song, try `+12`.

### 3. ❌ "Index file not found"
* **Solution:** Some models don't come with an `.index` file. The conversion will still work, but the voice might be less "accurate" to the original artist.

---